# Forecasting via Lag-Feature Attention
Thomas Keenan

**Table of Contents**


#### Imports and Modelling framework

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from pathlib import Path
from scipy import stats

sns.set_style("whitegrid")
seed = 40304451
np.random.seed(seed)

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor

In [13]:
tickers = ["XLK", "XLP", "XLV", "XLF", "XLE", "XLI"]
# Download each ticker
all_data = []
for ticker in tickers:
    stock = yf.Ticker(ticker)
    hist = stock.history(start="2010-01-01", end="2024-12-31", auto_adjust=False)
    hist['ticker'] = ticker
    hist = hist.reset_index()
    all_data.append(hist)
    
# Combine into one big df
prices_df = pd.concat(all_data, ignore_index=True)

# Change date column type
prices_df['Date'] = pd.to_datetime(prices_df['Date']).dt.tz_localize(None)

# Format column names for mltester
prices_df = prices_df.rename(columns={
    'Date': 'date',
    'Open': 'open', 
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'Adj Close': 'adj_close'
})

# Replace 0s with NaN in important cols
predictive_cols = ['open', 'high', 'low', 'close', 'volume']
prices_df[predictive_cols] = prices_df[predictive_cols].replace(0, np.nan)

# Set the date as the index and sort
prices_df = prices_df.set_index('date')
prices_df = prices_df.sort_values(['ticker', 'date'])

Missingness check:

In [15]:
print(prices_df.isna().sum())

open             0
high             0
low              0
close            0
adj_close        0
volume           0
Dividends        0
Stock Splits     0
Capital Gains    0
ticker           0
dtype: int64


There should be no missing values (as of December 2025)

### Forecaster Class:
The Forecaster class was developed as part of my original Forecasting H-Day cumulative log returns project, but has been streamlined given this study's scope.

It can take a JSON-style 'config' such as:
````python
configs = { 
    'task_1_model': {
        'n_lags': 15,
        'parkinson_vol_windows': (5, 10),
        'alpha_grid': (10.0, 50.0, 100.0)
    }
}
````
Which will test a Ridge model (the default model type) with strong L2 regularization using 15 day lagged returns (defaulted to 5) and parkinson volatility windows.

In [25]:
class Forecaster:
    '''Modelling class - input feature details and model architures. '''
    def __init__(
        self,
        config=None,
        random_state=seed,
    ):
        # Set defaults:
        
        # 5 lags was adopted in previous study
        self.n_lags = 5

        self.mom_windows = ()
        self.vol_window = None
        self.sma_windows = ()
        self.ema_windows = ()
        self.rsi_window = None
        self.parkinson_vol_windows = ()
        self.use_parkinson_vol_regime = False
        
        self.model_type = 'ridge'
        self.alpha_grid = (0.01, 0.1, 1.0, 10.0)
        self.cv_splits = 3
        self.min_train_points = 200
        self.random_state = random_state
                             
        # Override features with config details
        if isinstance(config, dict):
            for k, v in config.items():
                if hasattr(self, k):
                    setattr(self, k, v)

        # Convert types after config is applied
        self.n_lags = int(self.n_lags) if self.n_lags else 0
        self.mom_windows = tuple(int(w) for w in self.mom_windows) if self.mom_windows else ()
        self.vol_window = int(self.vol_window) if self.vol_window is not None else None
        self.sma_windows = tuple(int(w) for w in self.sma_windows) if self.sma_windows else ()
        self.ema_windows = tuple(int(w) for w in self.ema_windows) if self.ema_windows else ()
        self.rsi_window = int(self.rsi_window) if self.rsi_window is not None else None
        self.parkinson_vol_windows = tuple(int(w) for w in self.parkinson_vol_windows) if self.parkinson_vol_windows else None
        self.use_parkinson_vol_regime = bool(self.use_parkinson_vol_regime)
        
        self.model_type = str(self.model_type).lower()
        self.alpha_grid = tuple(float(a) for a in self.alpha_grid)
        self.cv_splits = int(self.cv_splits)
        self.min_train_points = int(self.min_train_points)
        self.random_state = int(self.random_state)
    
        self.pipe_ = None
        self.best_alpha_ = None
        self.fitted_ = False

    @staticmethod
    def _close_series(X: pd.DataFrame) -> pd.Series:
        return X["Close"] if "Close" in X.columns else X.iloc[:, 0]

    @staticmethod
    def _log_returns(series: pd.Series) -> pd.Series:
        series = pd.Series(series).astype(float)
        return np.log(series / series.shift(1))

    @staticmethod
    def _rsi(close: pd.Series, window: int) -> pd.Series:
        """RSI in [0,1] using Wilder's smoothing (causal)."""
        close = pd.Series(close).astype(float)
        diff = close.diff()
        gain = diff.clip(lower=0.0)
        loss = -diff.clip(upper=0.0)
        avg_gain = gain.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
        avg_loss = loss.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        rsi = 1 - (1 / (1 + rs))
        return rsi.fillna(0.5)

    @staticmethod
    def _garman_klass_vol(high: pd.Series, low: pd.Series, 
                          open_: pd.Series, close: pd.Series, 
                          window: int) -> pd.Series:
        """Garman-Klass volatility estimator"""
        high = pd.Series(high).astype(float)
        low = pd.Series(low).astype(float)
        open_ = pd.Series(open_).astype(float)
        close = pd.Series(close).astype(float)
        
        hl_ratio = (high / low).replace(0, np.nan)
        co_ratio = (close / open_).replace(0, np.nan)
        
        hl_term = 0.5 * np.log(hl_ratio) ** 2
        co_term = (2 * np.log(2) - 1) * np.log(co_ratio) ** 2
        
        gk = np.sqrt((hl_term - co_term).clip(lower=0))
        return gk.rolling(window, min_periods=window).mean()

    @staticmethod
    def _finite_mean(y: pd.Series) -> float:
        """Return finite mean of y or 0.0 if none available."""
        yv = pd.Series(y).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
        if len(yv) == 0:
            return 0.0
        m = float(yv.mean())
        return m if np.isfinite(m) else 0.0

    # Volume Regime Feature methods
    @staticmethod
    def _realized_volatility(returns: pd.Series, window: int) -> pd.Series:
        """Realized volatility: rolling standard deviation of log returns"""
        returns = pd.Series(returns).astype(float)
        return returns.rolling(window, min_periods=window).std(ddof=0)
    
    @staticmethod
    def _volatility_of_volatility(returns: pd.Series, window: int) -> pd.Series:
        """Volatility-of-volatility: rolling std of rolling std (matched windows)"""
        returns = pd.Series(returns).astype(float)
        realized_vol = returns.rolling(window, min_periods=window).std(ddof=0)
        vol_of_vol = realized_vol.rolling(window, min_periods=window).std(ddof=0)
        return vol_of_vol
    
    @staticmethod
    def _parkinson_volatility(high: pd.Series, low: pd.Series, window: int) -> pd.Series:
        high = pd.Series(high).astype(float)
        low = pd.Series(low).astype(float)
        
        hl_ratio = (high / low).replace(0, np.nan)
        squared_log = (np.log(hl_ratio) ** 2)
        
        parkinson_vol = np.sqrt(squared_log.rolling(window, min_periods=window).mean() / (4.0 * np.log(2)))
        
        return parkinson_vol

    
    @staticmethod
    def _volatility_ratio(returns: pd.Series, short_window: int, long_window: int) -> pd.Series:
        """Volatility ratio: short_vol / long_vol for regime change detection"""
        returns = pd.Series(returns).astype(float)
        
        short_vol = returns.rolling(short_window, min_periods=short_window).std(ddof=0)
        long_vol = returns.rolling(long_window, min_periods=long_window).std(ddof=0)
        
        ratio = short_vol / long_vol.replace(0, np.nan)
        return ratio.clip(0.1, 10.0)

    @staticmethod
    def _parkinson_vol_regime(returns: pd.Series, window: int, method='median') -> pd.Series:
        rv = returns.rolling(window, min_periods=window).std(ddof=0)
        
        if method == 'median':
            # 0 = low, 1 = high
            regime = (rv > rv.median()).astype(int) 
        else:
            raise ValueError("method must be 'median'")
        
        return regime.shift(1)


    def _make_features(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Build features based on config.
        """
        close = self._close_series(X).astype(float)
        lr = self._log_returns(close)
    
        feats = {}

        # Lags
        if self.n_lags > 0:
            for i in range(1, self.n_lags + 1):
                # Raw lags
                feats[f"lag{i}"] = lr.shift(i)
            
            # for w in windows_to_use:
            #     feats[f"realized_vol_{w}"] = self._realized_volatility(lr, w).shift(1)
        
        # Parkinson Volatility (needs OHLC)
        if self.parkinson_vol_windows is not None:
            if all(col in X.columns for col in ["High", "Low"]):
                windows_to_use = list(self.parkinson_vol_windows)
                
                for w in windows_to_use:
                    feats[f"parkinson_vol_{w}"] = self._parkinson_volatility(
                        X["High"], X["Low"], w
                    ).shift(1)

        # Parkinson-based Volatility Regime
        if self.use_parkinson_vol_regime and self.parkinson_vol_windows is not None:
            for w in self.parkinson_vol_windows:
                feats[f'parkinson_regime_{w}'] = self._parkinson_vol_regime(lr, w)

        # Momentum
        if self.mom_windows:
            for w in self.mom_windows:
                feats[f"mom_{w}"] = lr.rolling(w, min_periods=w).mean()
    
        if self.vol_window is not None: 
            feats[f"vol_{self.vol_window}"] = lr.rolling(self.vol_window, min_periods=self.vol_window).std(ddof=0)
    
        # SMA distances
        if self.sma_windows:
            for w in self.sma_windows:
                sma = close.rolling(w, min_periods=w).mean()
                feats[f"sma_dist_{w}"] = (close - sma) / sma.replace(0, np.nan)
    
        # EMA distances
        if self.ema_windows:
            for w in self.ema_windows:
                ema = close.ewm(span=w, adjust=False, min_periods=w).mean()
                feats[f"ema_dist_{w}"] = (close - ema) / ema.replace(0, np.nan)
    
        # RSI
        if self.rsi_window is not None:
            feats[f"rsi_{self.rsi_window}"] = self._rsi(close, self.rsi_window)
    
        # Return empty DataFrame if no features
        if not feats:
            return pd.DataFrame(index=X.index)
    
        F = pd.DataFrame(feats, index=X.index).replace([np.inf, -np.inf], np.nan)
        F = F.dropna()
        return F

    def _get_model(self, alpha=None):
        if self.model_type == 'somethingelse':
            return Ridge(alpha=alpha or 1.0, random_state=self.random_state)
        
        else:
            # The default model for this work is Ridge
            return Ridge(alpha=alpha or 1.0, random_state=self.random_state)

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series, meta=None):
        """
        Train with selected model architecture
        """
        F = self._make_features(X_train)

        if F.empty:
            mean_y = self._finite_mean(y_train)
            self.pipe_ = Pipeline([("model", DummyRegressor(strategy="constant", constant=mean_y))])
            self.pipe_.fit([[0.0]], [0.0])
            self.best_alpha_ = None
            self.fitted_ = True
            return self

        y = y_train.reindex(F.index)
        mask = y.replace([np.inf, -np.inf], np.nan).notna()
        F, y = F.loc[mask], y.loc[mask]

        if len(y) == 0:
            mean_y = self._finite_mean(y_train)
            self.pipe_ = Pipeline([("model", DummyRegressor(strategy="constant", constant=mean_y))])
            self.pipe_.fit([[0.0]], [0.0])
            self.best_alpha_ = None
            self.fitted_ = True
            return self

        if len(F) < self.min_train_points:
            mean_y = float(y.mean()) if np.isfinite(y.mean()) else 0.0
            self.pipe_ = Pipeline([("model", DummyRegressor(strategy="constant", constant=mean_y))])
            self.pipe_.fit([[0.0]], [0.0])
            self.best_alpha_ = None
            self.fitted_ = True
            return self

        # For linear models, tune alpha with scaling
        n_splits = min(self.cv_splits, max(2, len(F) // 200))
        tscv = TimeSeriesSplit(n_splits=n_splits)

        best_alpha, best_mse = None, np.inf
        for a in self.alpha_grid:
            mses = []
            for tr_idx, va_idx in tscv.split(F.values):
                X_tr, X_va = F.values[tr_idx], F.values[va_idx]
                y_tr, y_va = y.values[tr_idx], y.values[va_idx]
                
                model = self._get_model(alpha=a)
                pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
                pipe.fit(X_tr, y_tr)
                y_hat = pipe.predict(X_va)
                mses.append(mean_squared_error(y_va, y_hat))
            
            avg_mse = float(np.mean(mses)) if mses else np.inf
            if avg_mse < best_mse:
                best_mse, best_alpha = avg_mse, a

        if best_alpha is None:
            best_alpha = 1.0
        self.best_alpha_ = float(best_alpha)

        model = self._get_model(alpha=self.best_alpha_)
        self.pipe_ = Pipeline([("scaler", StandardScaler()), ("model", model)])
        self.pipe_.fit(F.values, y.values)
        self.fitted_ = True
        return self

    def predict(self, X: pd.DataFrame, meta=None) -> pd.Series:
        """
        Return predictions of next-H-day cumulative log returns.
        """
        F = self._make_features(X)
        if not self.fitted_ or self.pipe_ is None:
            idx = F.index if len(F) else X.index
            return pd.Series(0.0, index=idx, name="y_pred")
        if F.empty:
            return pd.Series(0.0, index=X.index, name="y_pred")
        y_hat = self.pipe_.predict(F.values)
        return pd.Series(y_hat, index=F.index, name="y_pred")

#### Walk Forward Validation
This method calls a forecasting model based on the configuration inputted, then evaluates it using walk-forward validation with an expanding training window. To respect the temporal characteristic of time series data the model is retained at each step on all historical data up to the test period, ensuring realistic out-of-sample predictions.

In [32]:
def walk_forward(prices, config, horizon=5, step=10):
    """
    Test a Forecasting configuration using walk-forward validation.
    """
    y_full = forward_log_return(prices["Close"], horizon=horizon)
    y_full.name = "y_true"
    test_dates = y_full.dropna().index
    
    all_preds = []
    
    for i in range(0, len(test_dates), step):
        block = test_dates[i : i + step]
        first_test = block[0]
        
        # Expanding window: train on everything before first test date
        X_train = prices.loc[: first_test - pd.Timedelta(days=1)]
        y_train = y_full.loc[: first_test - pd.Timedelta(days=1)]

        # Set a model based on the Forecaster configuration
        model = Forecaster(config=config)

        # Train
        model.fit(X_train, y_train, meta={"horizon": horizon})
        
        X_pred = prices.loc[: block[-1]]
        # Test
        y_hat = model.predict(X_pred, meta={"horizon": horizon})
        
        all_preds.append(y_hat.reindex(block).dropna())
    
    # Combine all predictions
    y_pred = pd.concat(all_preds) if all_preds else pd.Series(dtype=float, name="y_pred")
    y_true = y_full.reindex(y_pred.index)
    
    return y_true, y_pred

#### Run Forecasting Tests:

In [37]:
def run_evaluation(
    configs,
    prices_df,
    tickers,
    date_start,
    date_end,
    horizon,
    step,
    verbose=True,
    return_predictions=False
):
    """
    Evaluate multiple forecasting configurations across tickers using walk-forward validation.
    For each configuration and ticker combination, performs walk-forward validation and computes directional accuracy, MAE, and RMSE metrics.
    """
    results = []
    
    for config_name, config_params in configs.items():
        if verbose:
            print(f"\n{'_'*60}")
            print(f"Config: {config_name}")
            print(f"Config: {config_params}\n")
        
        for ticker in tickers:
            ticker_prices = get_ticker_data(prices_df, ticker)
            ticker_prices = ticker_prices.loc[date_start:date_end]
            
            y_true, y_pred = walk_forward(
                ticker_prices, 
                config_params, 
                horizon=horizon, 
                step=step
            )
            
            metrics = compute_metrics(y_true, y_pred)
            
            if return_predictions:
                for date, true_val, pred_val in zip(y_true.index, y_true.values, y_pred.values):
                    results.append({
                        'config': config_name,
                        'ticker': ticker,
                        'date': date,
                        'y_true': true_val,
                        'y_pred': pred_val,
                        'diracc': metrics['diracc'],
                        'mae': metrics['mae'],
                        'rmse': metrics['rmse'],
                    })
            else:
                results.append({
                    'config': config_name,
                    'ticker': ticker,
                    'diracc': metrics['diracc'],
                    'mae': metrics['mae'],
                    'rmse': metrics['rmse'],
                })
            
            if verbose:
                # Log individual ticker results
                print(f"{ticker}: DirAcc={metrics['diracc']:.4f}, "
                      f"MAE={metrics['mae']:.6f}, RMSE={metrics['rmse']:.6f}, ")
    
    results_df = pd.DataFrame(results)
    if verbose:
        print("\nTest Results:")
        print(results_df.groupby('config')[['diracc', 'mae', 'rmse']].mean().sort_values('diracc', ascending=False))
    
    return results_df